# SD 1.5 baseline: generation and evaluation (main_sd15)

This notebook runs the full SD 1.5 baseline on the main split (576 prompts × 4 seeds = 2,304 images): generation with Stable Diffusion 1.5, evaluation with OWLv2 and Qwen2.5-VL-7B, and metric analysis. Outputs are stored on Google Drive in `attribute_interference/v1/main_sd15`.

**Deviations from the project protocol (preliminary results):**
- **No evaluator freeze.** Main evaluation normally requires `evaluator_freeze.json` from the reviewed SDXL pilot, which is not available yet. The freeze check is skipped without modifying the project source files; the run is marked with `PRELIMINARY_UNFROZEN.json`.
- **Quantized Qwen.** Qwen2.5-VL-7B is loaded in 8-bit (bitsandbytes) so it fits entirely on a T4 GPU. The vision encoder and output head stay in FP16. Quantized answers are stored separately in `qwen_int8_records/` and results in `analysis_qwen_int8/`, so they never mix with a future frozen FP16 evaluation.
- OWLv2 runs unchanged (FP32).

**Run order:** Runtime → T4 GPU → cells 1–2 → restart session → cells 1, 3, 4 → remaining cells in order. All stages are resumable: after a disconnect, rerun cells 1, 3, 4 and the interrupted cell.

## 1. Repository
Clone the project repository. Generation was run on commit `0de4f3ab`; the run refuses to resume on a different source version.

In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO_URL = "https://github.com/IzyumovMaxim/cv_project"
REPO = Path("/content/cv_project")
if not (REPO / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print("Repository:", REPO)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

## 2. Dependencies
Install the pinned CUDA stack, the project package and bitsandbytes for 8-bit Qwen. Restart the session afterwards.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "bitsandbytes==0.47.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
print("Installed. Restart the session, then run cells 1, 3, 4.")

## 3. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

## 4. Google Drive and helpers
Mount Drive (outputs persist there) and define the pipeline helpers: `run` calls the project CLI, `evaluate` runs OWLv2 or the 8-bit Qwen evaluator, `analyze` computes metrics, `progress` reports evaluation coverage.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, re
OUTPUTS = Path("/content/drive/MyDrive/attribute_interference/v1")
OUTPUTS.mkdir(parents=True, exist_ok=True)
ROOT = OUTPUTS / "main_sd15"
CONFIG = REPO / "configs" / "main_sd15.yaml"
QUANT = "int8"

RUNNER = Path("/content/eval_quant.py")
RUNNER.write_text('''import sys, json, shutil, datetime
from pathlib import Path
import attribute_interference.storage as st
import attribute_interference.evaluation as ev
import attribute_interference.analysis as an

config, root, mode, quant = sys.argv[1], Path(sys.argv[2]), sys.argv[3], sys.argv[4]
assert mode in {'evaluate', 'analyze'} and quant in {'int8', 'nf4'}
cfg = st.load_config(config)
QDIR = f'qwen_{quant}_records'

_records, _put = st.records, st.put_record
def records(r, kind):
    return _records(r, QDIR if kind == 'qwen_records' else kind)
def put_record(r, kind, key, value):
    return _put(r, QDIR if kind == 'qwen_records' else kind, key, value)
ev.records = an.records = records
ev.put_record = put_record

def skip_freeze(cfg, root, frozen):
    marker = Path(root) / 'PRELIMINARY_UNFROZEN.json'
    if not marker.exists():
        st.atomic_json(marker, {
            'note': 'Evaluated WITHOUT a reviewed pilot evaluator freeze. Preliminary results only; '
                    'evaluator accuracy not validated against human labels.',
            'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')})
ev.ensure_freeze = skip_freeze

def quantized_init(self, cfg, root, torch):
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    self.torch = torch
    e = cfg['evaluation']
    revision = ev.model_revision(root, 'qwen', e['qwen_checkpoint'], e['qwen_revision'])
    skip = ['visual', 'lm_head']
    if quant == 'int8':
        q = BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=skip)
    else:
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16, llm_int8_skip_modules=skip)
    self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        e['qwen_checkpoint'], revision=revision, torch_dtype=torch.float16, attn_implementation='sdpa',
        device_map={'': 0}, quantization_config=q).eval()
    self.processor = AutoProcessor.from_pretrained(e['qwen_checkpoint'], revision=revision,
                                                   min_pixels=256*28*28, max_pixels=e['max_pixels'])
    self.max_tokens = e['max_new_tokens']
ev.QwenEvaluator.__init__ = quantized_init

try:
    if mode == 'evaluate':
        st.atomic_json(root / f'QWEN_{quant.upper()}_NOTE.json', {
            'note': f'Qwen2.5-VL-7B evaluated with bitsandbytes {quant} quantization (deviation from the '
                    'proposal, which specifies FP16 without quantization). Answers in ' + QDIR + '/.'})
        ev.evaluate(cfg, root, 'qwen')
    else:
        an.analyze(cfg, root)
        dest = root / f'analysis_qwen_{quant}'
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(root / 'analysis', dest)
        shutil.copy(root / 'labels.csv', dest / 'labels.csv')
        (dest / 'NOTE.txt').write_text(
            f'Preliminary results: Qwen quantized ({quant}), no reviewed evaluator freeze.\\n'
            'OWLv2 unchanged (FP32). Do not present as the final frozen evaluation.\\n')
        print('Results copied to', dest)
except (ValueError, RuntimeError, FileNotFoundError, ImportError) as exc:
    print(f'ERROR: {exc}', file=sys.stderr)
    sys.exit(1)
''')

OWL_CMD = """
import sys, attribute_interference.evaluation as ev
from attribute_interference.storage import load_config
ev.ensure_freeze = lambda *a: None
try:
    ev.evaluate(load_config(sys.argv[1]), sys.argv[2], "owl")
except (ValueError, RuntimeError, FileNotFoundError, ImportError) as exc:
    print("ERROR:", exc, file=sys.stderr); sys.exit(1)
"""

def run(command, config, name, *extra):
    subprocess.run([sys.executable, "-m", "attribute_interference.cli", command,
                    "--config", str(REPO / "configs" / (config + ".yaml")),
                    "--output", str(OUTPUTS / name), *map(str, extra)], check=True, cwd=REPO)

def stream(cmd):
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        m = re.search(r"\[(\d+)/(\d+)\].*: (\w+)$", line.strip())
        if m is None or m.group(3) != "ok" or int(m.group(1)) % 25 == 0 or m.group(1) == m.group(2):
            print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Command failed with exit code {p.returncode}; see the output above")

def evaluate(backend):
    if backend == "owl":
        stream([sys.executable, "-c", OWL_CMD, str(CONFIG), str(ROOT)])
    else:
        stream([sys.executable, str(RUNNER), str(CONFIG), str(ROOT), "evaluate", QUANT])

def analyze():
    stream([sys.executable, str(RUNNER), str(CONFIG), str(ROOT), "analyze", QUANT])

def progress():
    gen = [json.loads(p.read_text()) for p in (ROOT / "generation_records").glob("*.json")]
    print(f"Generated: {sum(r['status'] == 'ok' for r in gen)} images")
    for role in ["owl", f"qwen_{QUANT}"]:
        recs = [json.loads(p.read_text()) for p in (ROOT / f"{role}_records").glob("*.json")]
        ok = sum(r["status"] == "ok" for r in recs)
        print(f"{role}: evaluated {ok}, errors {len(recs) - ok}")
print("Outputs:", OUTPUTS)

## 5. Smoke test (8 images)
Quick SD 1.5 integration check before the full run.

In [ ]:
run("prepare", "smoke", "smoke_sd15")
run("generate", "smoke", "smoke_sd15")

## 6. SD 1.5 generation (2,304 images)
Generation is resumable: completed images are skipped on rerun. Images flagged by the safety checker are recorded as errors (3 images in our run).

In [ ]:
run("prepare", "main_sd15", "main_sd15")
run("generate", "main_sd15", "main_sd15")

## 7. Generation summary

In [ ]:
import statistics
from collections import Counter
recs = [json.loads(p.read_text()) for p in (ROOT / "generation_records").glob("*.json")]
print("Records:", len(recs), "of 2304")
print(Counter(r["status"] for r in recs))
for r in recs:
    if r["status"] != "ok":
        print(r["image_id"], "| seed", r["seed"], "|", r.get("error"))
secs = [r["seconds"] for r in recs if r["status"] == "ok"]
print("Mean generation time:", round(statistics.mean(secs), 2), "s")

## 8. OWLv2 evaluation
Object detection for prompts with a count (N) or position (P) constraint.

In [ ]:
evaluate("owl")

## 9. Qwen2.5-VL-7B evaluation (8-bit)
One templated question per requested factor plus object presence.

In [ ]:
evaluate("qwen")

## 10. Progress

In [ ]:
progress()

## 11. Metrics and figures
Results are copied to `main_sd15/analysis_qwen_int8/` together with `NOTE.txt`.

In [ ]:
analyze()
from attribute_interference.storage import read_csv
from IPython.display import display, Image as DisplayImage
import pandas as pd
OUT = ROOT / f"analysis_qwen_{QUANT}"
summary = json.loads((OUT / "summary.json").read_text())
print(json.dumps(summary["run"], indent=2))
display(pd.DataFrame(read_csv(OUT / "attributes.csv")))
display(pd.DataFrame(read_csv(OUT / "by_complexity.csv")))
for figure in ["attribute_accuracy.png", "complexity.png", "interference.png"]:
    display(DisplayImage(filename=str(OUT / figure)))

## Results
- `main_sd15/analysis_qwen_int8/`: `summary.json`, `attributes.csv`, `by_complexity.csv`, `interference.csv`, `singletons.csv`, `labels.csv`, figures, `failure_cases.html`, `NOTE.txt`.
- Report notes: 8-bit quantized Qwen, no evaluator freeze (preliminary), 3 images rejected by the SD 1.5 safety checker, T4 GPU.